# Proyecto de Detección de Fraude en Transacciones Bancarias

### 1. INTRODUCCION:

    Este proyecto tiene como objetivo desarrollar un modelo que supere 0.75 en las métricas AUC-PR, F1-score y F2-score para detectar transacciones fraudulentas en una entidad bancaria.

    Los datos provienen de Kaggle y contienen registros de transacciones etiquetadas como correctas o fraudulentas.
    El modelo debe predecir si una transacción es legítima o fraudulenta, ayudando a minimizar pérdidas económicas y proteger a los clientes.


### 2. Importacion de librerias y Configuracion

In [1]:
import os
import sys
sys.path.append(os.path.abspath("../"))
from src.data.data_clean import load_data, clean_data, save_data, drop_unnecessary_columns
from src.features.engineering import calculate_age, day_of_week, hour_of_day, distance_transaction
from src.preprocessing.preprocessing import apply_preprocessing
from src.modeling.models import run_all_models
from sklearn.model_selection import train_test_split

### 3. Carga y descripcion de los datos

In [2]:
# Carga de los datos
df = load_data("../data/raw/fraudTrain.csv")

df.head(5)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

* El dataset contiene variables como fecha y hora de la transacción, detalles del comercio, información del cliente, coordenadas geográficas, y la etiqueta de fraude.


In [4]:
#Limpiamos con nuestra funcion, descrita en eda.ipynb

df_cleaned = clean_data(df)

df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Unnamed: 0             1296675 non-null  int64         
 1   trans_date_trans_time  1296675 non-null  datetime64[ns]
 2   cc_num                 1296675 non-null  int64         
 3   merchant               1296675 non-null  object        
 4   category               1296675 non-null  object        
 5   amt                    1296675 non-null  float64       
 6   first                  1296675 non-null  object        
 7   last                   1296675 non-null  object        
 8   gender                 1296675 non-null  object        
 9   street                 1296675 non-null  object        
 10  city                   1296675 non-null  object        
 11  state                  1296675 non-null  object        
 12  zip                    12966

### 4. Análisis Exploratorio de Datos (EDA)

- Se identificaron variables que requerían conversión de tipo, como fechas y horas.
- Se verificaron duplicados y se eliminaron si existían.
- Se analizaron patrones temporales: horas y días con mayor incidencia de fraude.
- Se calculó la edad de los usuarios para analizar grupos etarios afectados:

  - Jóvenes (18-20) y mayores (60+) tienen menos transacciones pero mayor proporción de fraude.
  - Adultos (21-59) tienen más transacciones pero menor ratio de fraude.

- Se detectaron comercios con alta concentración de fraude, destacando 'Shopping_net' como la categoría más vulnerable.


![transactions_hour](figure/fig1_transactions_hour.png)

![transactions_week](figure/fig2_transaction_week.png)

![fraud_by_age](figure/fig3_fraud_byage.png)

![volumen_transaction](figure/fig4_volumen_transaction.png)

### 5. Preprocesamiento y Transformaciones

In [5]:
# Aplicar feature engineering
df_train_cleaned = calculate_age(df_cleaned)
df_train_cleaned = day_of_week(df_cleaned)
df_train_cleaned = hour_of_day(df_cleaned)
df_train_cleaned = distance_transaction(df_cleaned)
df_train_final = drop_unnecessary_columns(df_cleaned)

df_test = load_data("../data/raw/fraudTest.csv")
df_test_cleaned = clean_data(df_test)
df_test_cleaned = calculate_age(df_test_cleaned)
df_test_cleaned = day_of_week(df_test_cleaned)
df_test_cleaned = hour_of_day(df_test_cleaned)
df_test_cleaned = distance_transaction(df_test_cleaned)
df_test_final = drop_unnecessary_columns(df_test_cleaned)

In [6]:
#Si deseamos preparar los datos para el modelo, debemos cambiar la cantidad de datos de prueba y test

# ================ CONFIGURACIÓN MODO PRUEBA ================
TEST_MODE = True  # Cambiar a False para ejecución completa
SAMPLE_SIZE = 0.01  # 1% de datos para modo prueba

# Muestreo para modo prueba
if TEST_MODE:
    _, df_train_final, _, _ = train_test_split(
        df_train_final, 
        df_train_final['is_fraud'], 
        stratify=df_train_final['is_fraud'], 
        train_size=SAMPLE_SIZE,
        random_state=42
    )
    print(f"⚠️ MODO PRUEBA ACTIVADO: Usando {SAMPLE_SIZE*100}% de datos de entrenamiento")

    _, df_test_final, _, _ = train_test_split(
        df_test_final, 
        df_test_final['is_fraud'], 
        stratify=df_test_final['is_fraud'], 
        train_size=SAMPLE_SIZE,
        random_state=42
    )
    print(f"⚠️ MODO PRUEBA ACTIVADO: Usando {SAMPLE_SIZE*100}% de datos de prueba")


⚠️ MODO PRUEBA ACTIVADO: Usando 1.0% de datos de entrenamiento
⚠️ MODO PRUEBA ACTIVADO: Usando 1.0% de datos de prueba


In [7]:
# Definir columnas
categorical_columns = ['category', 'gender']
numerical_cols = ['amt', 'age', 'hour_of_day', 'day_of_week', 'distancia_km']
target_col = 'is_fraud'

# Aplicar preprocesamiento
X_train, y_train, X_test, y_test = apply_preprocessing(
    df_train=df_train_final,
    df_test=df_test_final,
    categorical_cols=categorical_columns,
    numerical_cols=numerical_cols,
    target_col=target_col
)

print(f"\n📊 Dimensiones finales de los datos:")
print(f"  Entrenamiento: {X_train.shape[0]} registros")
print(f"  Prueba: {X_test.shape[0]} registros")
print(f"  Fraudes en entrenamiento: {sum(y_train)} ({sum(y_train)/len(y_train):.4f}%)")
print(f"  Fraudes en prueba: {sum(y_test)} ({sum(y_test)/len(y_test):.4f}%)")


📊 Dimensiones finales de los datos:
  Entrenamiento: 1283709 registros
  Prueba: 550162 registros
  Fraudes en entrenamiento: 7431 (0.0058%)
  Fraudes en prueba: 2124 (0.0039%)


- Se crearon variables nuevas: edad, hora y día de la semana extraídos de la fecha.
- Se calculó la distancia entre la ubicación de la transacción y el comercio usando latitud y longitud.
- Se eliminaron columnas irrelevantes para el modelado:

  `"Unnamed: 0", "trans_date_trans_time", "merchant", "first", "last", "street",
  "city", "state", "zip", "city_pop", "job", "dob", "trans_num",
  "unix_time", "cc_num", "lat", "long", "merch_lat", "merch_long"`


### 6. Modelamiento

In [10]:
results_mode_test = run_all_models(X_train, y_train, X_test, y_test, test_mode=True)

# Mostrar resultados comparativos
print("\n🏆 RESULTADOS COMPARATIVOS FINALES:")
for model, metrics in results_mode_test.items():
    print(f"\n🔹 {model.upper()}:")
    print(f"   AUC-PR: {metrics['auc_pr']:.4f}")
    print(f"   F2-Score: {metrics['f2_score']:.4f}")
    print(f"   Mejores parámetros: {metrics['best_params']}")


⚖️ Desbalance de clases: 1:171.75

⚠️ MODO PRUEBA ACTIVADO (configuración rápida)

🚀 Comenzando entrenamiento de 3 modelos


Progreso General:   0%|          | 0/3 [00:00<?]


⚙️ Entrenando modelo: RF

🔧 Configuración para rf:
   Muestra: 1283709 registros
   Iteraciones: 5
   Folds validación: 2
   Parámetros a probar: 5 combinaciones
⏳ rf (1283709 muestras):   0%|          | 0/10 [05:01<?]


✅ RF completado:  33%|███▎      | 1/3 [05:03<10:07]]


✅ Modelo rf entrenado
📊 AUC-PR: 0.8749
🎯 F1-Score: 0.8272
🚨 F2-Score: 0.8045

⚙️ Entrenando modelo: RF_SMOTE

🔧 Configuración para rf_smote:
   Muestra: 1283709 registros
   Iteraciones: 5
   Folds validación: 2
   Parámetros a probar: 4 combinaciones
⏳ rf_smote (1283709 muestras):   0%|          | 0/10 [10:20<?]


...


✅ Modelo rf_smote entrenado
📊 AUC-PR: 0.8467
🎯 F1-Score: 0.6238
🚨 F2-Score: 0.7570

⚙️ Entrenando modelo: XGB

🔧 Configuración para xgb:
   Muestra: 1283709 registros
   Iteraciones: 5
   Folds validación: 2
   Parámetros a probar: 5 combinaciones


c:\Users\Paulo God\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:12:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


⏳ xgb (1283709 muestras):   0%|          | 0/10 [00:33<?]


✅ XGB completado: 100%|██████████| 3/3 [16:03<00:00]     


✅ Modelo xgb entrenado
📊 AUC-PR: 0.8671
🎯 F1-Score: 0.3420
🚨 F2-Score: 0.5573

🏆 RESULTADOS COMPARATIVOS FINALES:

🔹 RF:
   AUC-PR: 0.8749
   F2-Score: 0.8045
   Mejores parámetros: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}

🔹 RF_SMOTE:
   AUC-PR: 0.8467
   F2-Score: 0.7570
   Mejores parámetros: {'smote__k_neighbors': 3, 'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__max_depth': 20}

🔹 XGB:
   AUC-PR: 0.8671
   F2-Score: 0.5573
   Mejores parámetros: {'subsample': 0.8, 'reg_alpha': 0.1, 'max_depth': 7, 'learning_rate': 0.1, 'gamma': 0.2}


Se entrenaron tres modelos principales:

- Random Forest
- Random Forest con SMOTE para balanceo de clases
- XGBoost con búsqueda de hiperparámetros usando Random Search y Stratified K-Fold

Resultados obtenidos:


![resultados_finales](figure/comparacion_modelos_metricas.png)


### 7. Resultados y Evaluación

El modelo Random Forest fue el mejor, superando el umbral de 0.75 en todas las métricas clave.

El modelo con SMOTE no mejoró el desempeño, posiblemente por el ruido generado en el oversampling.

XGBoost mostró bajo rendimiento en F1 y F2, aunque se recomienda seguir ajustando hiperparámetros para mejorar.

Se concluye que Random Forest es el modelo más adecuado para la detección de fraude en este caso.

### 8. Conclusiones y Recomendaciones

- Random Forest es el modelo recomendado para implementar en producción.
- Se sugiere continuar alimentando el modelo con nuevos datos para mejorar su desempeño.
- Es preferible prevenir bloqueando transacciones sospechosas, incluso si esto genera inconvenientes menores a clientes legítimos, para evitar pérdidas por fraude.
- Explorar otras técnicas de balanceo y modelos podría mejorar aún más los resultados.
